In [1]:
import torch
from torch._library.opaque_object import MemberType, OpaqueBase, register_opaque_type


# --- Opaque object with inlined own_setup_context ---

class ScaledOpConfig(OpaqueBase):
    def __init__(self, scale: float):
        self.scale = scale

    def own_setup_context(self, inputs, output, ctx): # This is marked as INLINED, but it is not inlined, it can do basically everything and is not traced(?)
        # i want functions marked as INLINED to be basically inlined - treated as the code was copy pasted.
        x = inputs[0]
        print("123")
        ctx.save_for_backward(x)
        ctx.cfg = self
    
    def custom_call(self, x): # This is not marked as USE_REAL/inline, so there will be error.
        return


register_opaque_type(
    ScaledOpConfig,
    typ="reference",
    members={
        "own_setup_context": MemberType.INLINED,
        "scale": MemberType.USE_REAL,
    },
)

opaque_type_name = torch._library.opaque_object.get_opaque_type_name(ScaledOpConfig)
lib = torch.library.Library("inline_lib", "DEF")

# --- Forward custom op ---

torch.library.define(
    "inline_lib::scaled_op",
    f"(Tensor x, {opaque_type_name} cfg) -> Tensor",
    tags=torch.Tag.pt2_compliant_tag,
    lib=lib,
)


@torch.library.impl("inline_lib::scaled_op", "CompositeExplicitAutograd", lib=lib)
def scaled_op_impl(x: torch.Tensor, cfg: ScaledOpConfig) -> torch.Tensor:
    return x * cfg.scale


@torch.library.register_fake("inline_lib::scaled_op", lib=lib)
def scaled_op_fake(x: torch.Tensor, cfg: ScaledOpConfig) -> torch.Tensor:
    return torch.empty_like(x)


# --- Backward custom op ---

torch.library.define(
    "inline_lib::scaled_op_backward",
    f"(Tensor grad_output, Tensor x, {opaque_type_name} cfg) -> Tensor",
    tags=torch.Tag.pt2_compliant_tag,
    lib=lib,
)


@torch.library.impl(
    "inline_lib::scaled_op_backward", "CompositeExplicitAutograd", lib=lib
)
def scaled_op_backward_impl(
    grad_output: torch.Tensor, x: torch.Tensor, cfg: ScaledOpConfig
) -> torch.Tensor:
    return grad_output * cfg.scale


@torch.library.register_fake("inline_lib::scaled_op_backward", lib=lib)
def scaled_op_backward_fake(
    grad_output: torch.Tensor, x: torch.Tensor, cfg: ScaledOpConfig
) -> torch.Tensor:
    return torch.empty_like(grad_output)


# --- Autograd: setup_context delegates to opaque object ---

def setup_context(ctx, inputs, output):
    x, cfg = inputs
    cfg.own_setup_context(inputs, output, ctx)
    cfg.custom_call(x) # you can remove it and it will pass


def backward(ctx, grad_output):
    (x,) = ctx.saved_tensors
    grad_x = torch.ops.inline_lib.scaled_op_backward(grad_output, x, ctx.cfg)
    return grad_x, None


torch.library.register_autograd(
    "inline_lib::scaled_op",
    backward,
    setup_context=setup_context,
    lib=lib,
)


# --- Tests ---


def test_compile_fullgraph():
    print("Test: torch.compile(fullgraph=True)")
    cfg = ScaledOpConfig(scale=2.0)
    x = torch.randn(8, requires_grad=True)

    @torch.compile(fullgraph=True)
    def fn(x, cfg):
        return torch.ops.inline_lib.scaled_op(x, cfg)

    y = fn(x, cfg)
    y.sum().backward()
    print(f"  y.shape = {y.shape}")
    print(f"  x.grad  = {x.grad}")
    assert torch.allclose(x.grad, torch.full_like(x, 2.0))
    print("  PASS")


if __name__ == "__main__":
    test_compile_fullgraph()
    print("\nAll tests passed.")


Test: torch.compile(fullgraph=True)
123


TorchRuntimeError: RuntimeError when making fake tensor call
  Explanation: Dynamo failed to run FX node with fake tensors: call_function inline_lib.scaled_op(*(FakeTensor(..., size=(8,), requires_grad=True), <torch._library.fake_class_registry.FakeScriptObject object at 0x7ca8b00764b0>), **{}): got AttributeError("Tried to call __getattr__ with attr 'custom_call' on a FakeScriptObject, implying that you are calling this inside of a fake kernel. The fake kernel should not depend on the contents of the OpaqueObject at all, so we're erroring out. If this attr is a method or constant attribute, you can allow this member access by registering it via `register_opaque_type(members=...)`.")
  Hint: Your code may result in an error when running in eager. Please double check that your code doesn't contain a similar error when actually running eager/uncompiled. You can do this by removing the `torch.compile` call, or by using `torch.compiler.set_stance("force_eager")`. 

  Developer debug contex